# N-ATLAS Forge: run the official N-ATLaS worker on a free GPU

This notebook starts the worker from `inference/natlas-worker` in the Forge repository, loads the
official `NCAIR1/N-ATLaS` weights from Hugging Face, and publishes the worker through a temporary
HTTPS address so the Forge backend on your PC can use it. Nothing here replaces or imitates the model.

Before you run it (Kaggle):

1. Settings menu: Accelerator = **GPU T4 x2**. Right-hand panel (Notebook), Session options: Internet = **On**. Your Kaggle account must be phone verified.
2. Add-ons, Secrets: create two secrets and attach them to this notebook.
   `HF_TOKEN` is a Hugging Face read token from the account that accepted the terms at https://huggingface.co/NCAIR1/N-ATLaS.
   `NATLAS_WORKER_TOKEN` is the value of `NATLAS_WORKER_TOKEN` in `inference\natlas-worker\.env` on your PC.
3. Keep the notebook private. Run the cells in order. The last cell keeps running; leave it running and keep this tab open.

Every time you run the notebook again, the public address changes. Run `connect-worker.ps1` with the new address and restart the backend.

In [ ]:
# 1. Check the GPUs. The 8B model needs about 16 GB for its weights in float16, plus working memory.
import shutil
import subprocess


def parse_gpus(csv_text):
    gpus = []
    for line in csv_text.strip().splitlines():
        name, memory_mib = [part.strip() for part in line.rsplit(",", 1)]
        gpus.append((name, int(memory_mib)))
    return gpus


def gpu_is_enough(gpus):
    total = sum(memory for _, memory in gpus)
    biggest = max((memory for _, memory in gpus), default=0)
    return biggest >= 20 * 1024 or (len(gpus) >= 2 and total >= 28 * 1024)


try:
    smi = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
        capture_output=True,
        text=True,
    )
except FileNotFoundError:
    smi = None

if smi is None or smi.returncode != 0:
    raise SystemExit("No GPU found. In the Settings menu set Accelerator to GPU T4 x2, then run this notebook again.")

gpus = parse_gpus(smi.stdout)
for index, (name, memory_mib) in enumerate(gpus):
    print(f"GPU {index}: {name}, {memory_mib / 1024:.1f} GiB")

if not gpu_is_enough(gpus):
    raise SystemExit(
        "This GPU setup is too small. Choose GPU T4 x2 (two GPUs, about 30 GiB in total) "
        "or a single GPU with 24 GiB or more. A single 16 GiB GPU cannot hold the float16 weights."
    )

free_gib = shutil.disk_usage("/").free / 2**30
print(f"Free disk space: {free_gib:.0f} GiB (the download needs about 17 GiB)")
if free_gib < 25:
    print("Warning: free disk space is low. The model download may fail.")

In [ ]:
# 2. Install the worker's libraries. PyTorch already ships in the notebook image with CUDA, so it is left alone.
import subprocess
import sys
import urllib.request

print("Python", sys.version.split()[0], "at", sys.executable)

# Internet must be On, otherwise nothing can be installed or downloaded.
try:
    urllib.request.urlopen("https://pypi.org/simple/pip/", timeout=15).close()
except Exception as exc:
    raise SystemExit(
        "This session cannot reach the internet (" + type(exc).__name__ + "). Turn Internet On: in the right-hand "
        "panel (Notebook), scroll to Session options and switch Internet to On, then run this cell again."
    ) from None


def run_pip(*arguments):
    result = subprocess.run([sys.executable, "-m", "pip", *arguments], capture_output=True, text=True)
    return result.returncode, (result.stdout or "") + (result.stderr or "")


def pip_install(*requirements):
    code, output = run_pip("install", "-q", *requirements)
    if code != 0 and "No module named pip" in output:
        subprocess.run([sys.executable, "-m", "ensurepip", "--upgrade"], capture_output=True, text=True)
        code, output = run_pip("install", "-q", *requirements)
    if code != 0 and "externally-managed-environment" in output:
        # This image protects its system Python. The notebook machine is disposable, so allow the install.
        code, output = run_pip("install", "-q", "--break-system-packages", *requirements)
    if code != 0:
        print("\n".join(output.strip().splitlines()[-30:]))
        raise SystemExit("pip could not install " + ", ".join(requirements) + ". Its last output lines are shown above.")


pip_install("fastapi>=0.115,<1.0", "uvicorn[standard]>=0.30,<1.0", "pydantic>=2.8,<3.0", "pydantic-settings>=2.4,<3.0")
pip_install("transformers>=4.45,<6.0", "accelerate>=0.34", "huggingface_hub")

versions = subprocess.run(
    [
        sys.executable,
        "-c",
        "import torch, transformers, accelerate; "
        "print('torch', torch.__version__, '| transformers', transformers.__version__, '| accelerate', accelerate.__version__); "
        "print('CUDA available:', torch.cuda.is_available(), '| GPUs seen by PyTorch:', torch.cuda.device_count())",
    ],
    capture_output=True,
    text=True,
)
print((versions.stdout or versions.stderr).strip())
if "CUDA available: True" not in versions.stdout:
    raise SystemExit("PyTorch cannot see the GPU. Check the Accelerator setting and restart the session.")

In [ ]:
# 3. Read the secrets and confirm that Hugging Face lets this account download the gated model.
import os


def read_secret(name):
    value = os.environ.get(name)
    if value:
        return value.strip()
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret(name).strip()
    except Exception:
        pass
    try:
        from google.colab import userdata

        return userdata.get(name).strip()
    except Exception:
        return None


HF_TOKEN = read_secret("HF_TOKEN")
WORKER_TOKEN = read_secret("NATLAS_WORKER_TOKEN")

problems = []
if not HF_TOKEN or not HF_TOKEN.startswith("hf_"):
    problems.append("HF_TOKEN is missing or does not start with hf_. Create a read token at https://huggingface.co/settings/tokens and attach the secret to this notebook.")
if not WORKER_TOKEN or len(WORKER_TOKEN) < 32 or WORKER_TOKEN.startswith("CHANGE_ME"):
    problems.append("NATLAS_WORKER_TOKEN is missing or shorter than 32 characters. Copy it from inference\\natlas-worker\\.env on your PC.")
if problems:
    raise SystemExit("\n".join(problems))
print("Both secrets are present (their values are never printed).")

from huggingface_hub import hf_hub_download

try:
    hf_hub_download("NCAIR1/N-ATLaS", "config.json", token=HF_TOKEN)
    print("Hugging Face access to NCAIR1/N-ATLaS: OK")
except Exception as exc:
    kind = type(exc).__name__
    text = str(exc)
    if "Gated" in kind or "401" in text or "403" in text:
        raise SystemExit(
            "Hugging Face did not allow this token to read NCAIR1/N-ATLaS. Open https://huggingface.co/NCAIR1/N-ATLaS "
            "while signed in, accept the terms, wait if the page says your request is pending, and create the token "
            "in that same account."
        ) from None
    raise SystemExit(f"Could not reach Hugging Face ({kind}). Check that Internet is On (right-hand panel, Session options).") from None

In [ ]:
# 4. Fetch the worker code from the Forge repository (public). For a private fork, clone with a token instead.
import os
import subprocess

REPO_URL = "https://github.com/PereTheDataAnalyst/forge"
BRANCH = "main"
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
REPO = os.path.join(WORK, "forge")

if os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO], check=True)

WORKER_DIR = os.path.join(REPO, "inference", "natlas-worker")
if not os.path.isfile(os.path.join(WORKER_DIR, "app", "main.py")):
    raise SystemExit(f"The worker code was not found in {WORKER_DIR}.")
print("Worker code:", WORKER_DIR)

In [ ]:
# 5. Start the worker. float16 is used because T4 GPUs have no native bfloat16, and device_map=auto
#    splits the layers across both GPUs. Generation is capped at 90 s so a reply always finishes
#    before the 100 s limit of the tunnel.
import atexit
import os
import subprocess
import sys

PORT = 8081
LOG_PATH = os.path.join(WORK, "natlas-worker.log")

try:
    if worker.poll() is None:
        worker.terminate()
        worker.wait(timeout=30)
except NameError:
    pass

worker_env = os.environ.copy()
worker_env.update(
    {
        "NATLAS_WORKER_TOKEN": WORKER_TOKEN,
        "NATLAS_WORKER_MODEL_ID": "NCAIR1/N-ATLaS",
        "NATLAS_WORKER_DTYPE": "float16",
        "NATLAS_WORKER_DEVICE_MAP": "auto",
        "NATLAS_WORKER_MAX_GENERATION_SECONDS": "90",
        "HF_TOKEN": HF_TOKEN,
        "HF_HUB_DISABLE_TELEMETRY": "1",
        "TOKENIZERS_PARALLELISM": "false",
    }
)

log_file = open(LOG_PATH, "w")
worker = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:create_app", "--factory", "--host", "127.0.0.1", "--port", str(PORT)],
    cwd=WORKER_DIR,
    env=worker_env,
    stdout=log_file,
    stderr=subprocess.STDOUT,
)
atexit.register(lambda: worker.poll() is None and worker.terminate())
print("Worker process started (pid", worker.pid, "). Its log is", LOG_PATH)

In [ ]:
# 6. Wait until the model is loaded. The first run downloads about 16 GB, which usually takes several minutes.
import json
import subprocess
import time
import urllib.request

BASE = f"http://127.0.0.1:{PORT}"


def get_info():
    request = urllib.request.Request(BASE + "/v1/info", headers={"Authorization": f"Bearer {WORKER_TOKEN}"})
    with urllib.request.urlopen(request, timeout=10) as response:
        return json.loads(response.read().decode("utf-8"))


def log_tail(lines=40):
    with open(LOG_PATH, errors="replace") as handle:
        return "".join(handle.readlines()[-lines:])


deadline = time.time() + 30 * 60
last_state = None
info = None
while time.time() < deadline:
    if worker.poll() is not None:
        print(log_tail())
        raise SystemExit(f"The worker exited with code {worker.returncode}. The log above shows why.")
    try:
        info = get_info()
        state = info.get("state")
    except Exception:
        info, state = None, "starting"
    if state != last_state:
        print(time.strftime("%H:%M:%S"), "state:", state)
        last_state = state
    if state == "ready":
        break
    if state == "failed":
        print(log_tail())
        raise SystemExit("The model failed to load: " + str(info.get("error")))
    time.sleep(10)
else:
    print(log_tail())
    raise SystemExit("Timed out after 30 minutes while waiting for the model.")

print(json.dumps(info, indent=2))
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total", "--format=csv"])

In [ ]:
# 7. Ask the real model one question directly on the worker. This is real output from NCAIR1/N-ATLaS.
import json
import urllib.error
import urllib.request


def chat(messages, **parameters):
    body = {"messages": messages, "max_new_tokens": 128, "temperature": 0.1, "top_p": 1.0, "repetition_penalty": 1.12}
    body.update(parameters)
    request = urllib.request.Request(
        BASE + "/v1/chat",
        data=json.dumps(body).encode("utf-8"),
        headers={"Authorization": f"Bearer {WORKER_TOKEN}", "Content-Type": "application/json; charset=utf-8"},
        method="POST",
    )
    try:
        with urllib.request.urlopen(request, timeout=180) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as exc:
        raise SystemExit(f"Worker returned HTTP {exc.code}: {exc.read().decode('utf-8', 'replace')[:500]}") from None


result = chat([{"role": "user", "content": "Write one short greeting in English, Yoruba, Hausa and Igbo."}])
print(result["text"])
print({key: value for key, value in result.items() if key != "text"})

In [ ]:
# 8. Publish the worker through a Cloudflare quick tunnel (no account needed). The address is temporary.
#    Quick tunnels are meant for testing and do not stream server-sent events. The worker stays protected
#    by the bearer token. Move it to a private network once official compute is available.
import json
import os
import re
import stat
import subprocess
import threading
import time
import urllib.request

CLOUDFLARED = "/tmp/cloudflared"
if not os.path.exists(CLOUDFLARED):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        CLOUDFLARED,
    )
    os.chmod(CLOUDFLARED, os.stat(CLOUDFLARED).st_mode | stat.S_IEXEC)

URL_PATTERN = re.compile(r"https://[a-z0-9]+(?:-[a-z0-9]+)+\.trycloudflare\.com")


def start_tunnel(port):
    process = subprocess.Popen(
        [CLOUDFLARED, "tunnel", "--no-autoupdate", "--protocol", "http2", "--url", f"http://127.0.0.1:{port}"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    found = {"url": None}
    log = open("/tmp/cloudflared.log", "w")

    def pump():
        for line in process.stdout:
            log.write(line)
            log.flush()
            if found["url"] is None:
                match = URL_PATTERN.search(line)
                if match:
                    found["url"] = match.group(0)

    threading.Thread(target=pump, daemon=True).start()
    return process, found


try:
    if tunnel.poll() is None:
        tunnel.terminate()
        tunnel.wait(timeout=15)
except NameError:
    pass

tunnel, tunnel_state = start_tunnel(PORT)
deadline = time.time() + 90
while tunnel_state["url"] is None and time.time() < deadline and tunnel.poll() is None:
    time.sleep(1)

public_url = tunnel_state["url"]
if public_url is None:
    with open("/tmp/cloudflared.log", errors="replace") as handle:
        print(handle.read()[-2000:])
    raise SystemExit("The tunnel did not start. The log above shows why.")


def open_url(path, headers=None):
    merged = {"User-Agent": "natlas-forge-notebook/1.0"}
    merged.update(headers or {})
    return urllib.request.urlopen(urllib.request.Request(public_url + path, headers=merged), timeout=20)


for _ in range(30):
    try:
        with open_url("/healthz") as response:
            if response.status == 200:
                break
    except Exception:
        pass
    time.sleep(4)
else:
    print("The address does not answer yet. DNS can take a minute. Try", public_url + "/healthz", "in a browser.")

try:
    with open_url("/v1/info", {"Authorization": f"Bearer {WORKER_TOKEN}"}) as response:
        print("Through the tunnel, the worker reports state:", json.loads(response.read().decode("utf-8")).get("state"))
except Exception as exc:
    print("The authenticated check through the tunnel failed:", type(exc).__name__)

print()
print("Worker address:", public_url)
print()
print("On your PC, in PowerShell, from C:\\Projects\\forge, run:")
print("  .\\scripts\\connect-worker.ps1 -Url " + public_url)
print("Then close the backend window, run .\\scripts\\dev.ps1 again, and run .\\scripts\\smoke-day1.ps1.")

In [ ]:
# 9. Keep everything running. Leave this cell running and keep the tab open. Stop it (square button) to shut down.
import time

print("Running. Worker address:", public_url)
try:
    while True:
        time.sleep(300)
        problems = []
        if worker.poll() is not None:
            problems.append("the worker exited")
        if tunnel.poll() is not None:
            problems.append("the tunnel exited")
        try:
            state = get_info().get("state")
        except Exception as exc:
            state = "unreachable (" + type(exc).__name__ + ")"
        print(time.strftime("%H:%M:%S"), "worker:", state)
        if problems:
            raise SystemExit(", ".join(problems) + ". Run the notebook again. The public address will change.")
except KeyboardInterrupt:
    print("Stopped.")
finally:
    for process in (tunnel, worker):
        if process.poll() is None:
            process.terminate()